# 12-00 TabPFN Intro und Setup

Dieses Notebook prueft die gemeinsame TabPFN-Datenbasis fuer die finale Classifier-only-Pipeline.

Kernregeln:
- Drei binaere Targets: `Top5`, `Top10`, `Top20`.
- Nur `TabPFNClassifier`.
- Hyperparameterwahl spaeter nur ueber `Top10` und `roc_auc`.
- Finale Rankings spaeter nur ueber `score_topk_raw_sum = p_top5_raw + p_top10_raw + p_top20_raw`.
- Dieses Notebook startet keine API-Laeufe.


In [1]:
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "model_data").exists() and (candidate / "src" / "Notebooks").exists():
            return candidate
    raise FileNotFoundError("Projektwurzel mit data/model_data und src/Notebooks nicht gefunden.")

PROJECT_ROOT = find_project_root()
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_data"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABPFN_DIR = PROCESSED_DIR / "tabpfn"
LEGACY_DIR = PROCESSED_DIR / "tabpfn_3_experiments"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"MODEL_DATA_DIR: {MODEL_DATA_DIR}")


PROJECT_ROOT: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction
MODEL_DATA_DIR: /Users/roberthendrich/GADA-Group3-Cycling-Stage-Prediction/data/model_data


In [2]:
FEATURE_COLUMNS = [
    "distance",
    "vertical_meters",
    "stage_nr",
    "team_tier",
    "age_at_race",
    "rider_bmi",
    "wind_stability_index",
    "weather_temp_mean",
    "weather_temp_trend",
    "weather_rain_prob_mean",
    "weather_precipitation_mean",
    "weather_humidity_mean",
    "gradient_final_km",
    "lag_rider_points_season",
    "lag_rider_rank_season",
    "lag_race_competitiveness_median",
    "lag_team_power_index",
]

TARGET_CONFIGS = {
    "top5": {
        "label": "Top5",
        "train_file": "y_top5_train.pkl",
        "valid_file": "y_top5_valid.pkl",
        "test_file": "y_top5_test.pkl",
        "score_col": "p_top5_raw",
        "actual_col": "actual_top5",
    },
    "top10": {
        "label": "Top10",
        "train_file": "y_top10_train.pkl",
        "valid_file": "y_top10_valid.pkl",
        "test_file": "y_top10_test.pkl",
        "score_col": "p_top10_raw",
        "actual_col": "actual_top10",
    },
    "top20": {
        "label": "Top20",
        "train_file": "y_top20_train.pkl",
        "valid_file": "y_top20_valid.pkl",
        "test_file": "y_top20_test.pkl",
        "score_col": "p_top20_raw",
        "actual_col": "actual_top20",
    },
}

EXPECTED_SHAPES = {
    "train": (169349, 17),
    "valid": (8897, 17),
    "test": (17802, 17),
}


## Daten laden

Die Pickles werden aus `data/model_data/` geladen. Die Pfade werden ueber die Projektwurzel bestimmt, damit das Notebook nicht von einem bestimmten Arbeitsverzeichnis abhaengt.


In [3]:
SETUP_DIR = TABPFN_DIR / "00_setup"
SETUP_DIR.mkdir(parents=True, exist_ok=True)


def read_pickle(name):
    path = MODEL_DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_pickle(path)

X_train = read_pickle("X_train.pkl")
X_valid = read_pickle("X_valid.pkl")
X_test = read_pickle("X_test.pkl")

groups_train = read_pickle("groups_train.pkl")
groups_valid = read_pickle("groups_valid.pkl")
groups_test = read_pickle("groups_test.pkl")

meta_valid = read_pickle("meta_valid.pkl")
meta_test = read_pickle("meta_test.pkl")

y_rank_train = read_pickle("y_rank_train.pkl")
y_rank_valid = read_pickle("y_rank_valid.pkl")
y_rank_test = read_pickle("y_rank_test.pkl")

y_targets = {}
for target, cfg in TARGET_CONFIGS.items():
    y_targets[(target, "train")] = read_pickle(cfg["train_file"])
    y_targets[(target, "valid")] = read_pickle(cfg["valid_file"])
    y_targets[(target, "test")] = read_pickle(cfg["test_file"])

print("Loaded model data")
print("X_train", X_train.shape)
print("X_valid", X_valid.shape)
print("X_test", X_test.shape)


Loaded model data
X_train (169349, 17)
X_valid (8897, 17)
X_test (17802, 17)


## Feature- und Split-Checks

Die erwarteten Zeilenzahlen werden als Warnung behandelt. So bleibt das Notebook nutzbar, falls `10-00_Model_Data_Prep.ipynb` spaeter neu exportiert wird.


In [4]:
assert len(FEATURE_COLUMNS) == 17
assert list(X_train.columns) == FEATURE_COLUMNS
assert list(X_valid.columns) == FEATURE_COLUMNS
assert list(X_test.columns) == FEATURE_COLUMNS

for split_name, X in [("train", X_train), ("valid", X_valid), ("test", X_test)]:
    expected = EXPECTED_SHAPES[split_name]
    if X.shape != expected:
        warnings.warn(f"{split_name}: erwartete Shape {expected}, gefunden {X.shape}")

assert set(meta_valid["meta_year"].unique()) == {2023}
assert set(meta_test["meta_year"].unique()).issubset({2024, 2025})
assert len(groups_train) == len(X_train)
assert len(groups_valid) == len(X_valid)
assert len(groups_test) == len(X_test)

feature_columns = pd.DataFrame({"feature": FEATURE_COLUMNS, "position": range(len(FEATURE_COLUMNS))})
feature_columns.to_csv(SETUP_DIR / "feature_columns.csv", index=False)
feature_columns


,feature,position
0,distance,0
1,vertical_meters,1
2,stage_nr,2
3,team_tier,3
4,age_at_race,4
5,rider_bmi,5
6,wind_stability_index,6
7,weather_temp_mean,7
8,weather_temp_trend,8
9,weather_rain_prob_mean,9


## Split-Uebersicht schreiben


In [5]:
def split_overview_row(split, X, groups, meta=None, years_label=None):
    years = years_label
    if meta is not None and "meta_year" in meta.columns:
        years = ", ".join(map(str, sorted(pd.Series(meta["meta_year"]).dropna().unique())))
    return {
        "split": split,
        "rows": len(X),
        "features": X.shape[1],
        "stages": pd.Series(groups).nunique(),
        "years": years,
    }

split_overview = pd.DataFrame([
    split_overview_row("train_selection", X_train, groups_train, years_label="<=2022"),
    split_overview_row("validation", X_valid, groups_valid, meta_valid),
    split_overview_row("test_original", X_test, groups_test, meta_test),
    split_overview_row("train_final", pd.concat([X_train, X_valid], axis=0), pd.concat([groups_train, groups_valid], axis=0), years_label="<=2023"),
])

split_overview.to_csv(SETUP_DIR / "model_data_split_overview.csv", index=False)
split_overview


,split,rows,features,stages,years
0,train_selection,169349,17,997,<=2022
1,validation,8897,17,57,2023
2,test_original,17802,17,112,"2024, 2025"
3,train_final,178246,17,1054,<=2023


## Target-Verteilungen schreiben


In [6]:
def target_distribution_row(split, target, y, groups=None, meta=None, years_label=None):
    y = pd.Series(y)
    positives = int((y == 1).sum())
    years = years_label
    if meta is not None and "meta_year" in meta.columns:
        years = ", ".join(map(str, sorted(pd.Series(meta["meta_year"]).dropna().unique())))
    return {
        "split": split,
        "target": target,
        "rows": len(y),
        "positives": positives,
        "negatives": int((y == 0).sum()),
        "positive_rate": positives / len(y) if len(y) else np.nan,
        "stages": pd.Series(groups).nunique() if groups is not None else np.nan,
        "years": years,
    }

rows = []
for target in TARGET_CONFIGS:
    rows.append(target_distribution_row("train", target, y_targets[(target, "train")], groups_train, years_label="<=2022"))
    rows.append(target_distribution_row("validation", target, y_targets[(target, "valid")], groups_valid, meta_valid))
    rows.append(target_distribution_row("test", target, y_targets[(target, "test")], groups_test, meta_test))

target_distribution = pd.DataFrame(rows)
target_distribution.to_csv(SETUP_DIR / "target_distribution_by_split.csv", index=False)
target_distribution


,split,target,rows,positives,negatives,positive_rate,stages,years
0,train,top5,169349,5011,164338,0.029590,997,<=2022
1,validation,top5,8897,284,8613,0.031921,57,2023
2,test,top5,17802,556,17246,0.031232,112,"2024, 2025"
3,train,top10,169349,10006,159343,0.059085,997,<=2022
4,validation,top10,8897,570,8327,0.064067,57,2023
5,test,top10,17802,1104,16698,0.062016,112,"2024, 2025"
6,train,top20,169349,19965,149384,0.117893,997,<=2022
7,validation,top20,8897,1138,7759,0.127908,57,2023
8,test,top20,17802,2205,15597,0.123862,112,"2024, 2025"


## Legacy-Artefakte einordnen

Diese Artefakte sind historische Referenzen. Sie duerfen gelesen, aber nicht als neue Modellwahl verwendet werden.


In [7]:
legacy_artifacts = pd.DataFrame([
    {"artifact": "src/Notebooks/12-00-01_Versuch_1_TabPFN_3.ipynb", "role": "historische lokale Top10-Tests"},
    {"artifact": "src/Notebooks/12-00-02_tabpfn-client_Versuch.ipynb", "role": "historischer API-Versuch"},
    {"artifact": "src/Notebooks/tabpfn_alt/12-02-02_TabPFN_Client_Learning_Curve_Validation_2023.ipynb", "role": "Legacy-Learning-Curve, nur Kontext"},
    {"artifact": "src/Notebooks/tabpfn_alt/12-03-01_TabPFN_Classifier.ipynb", "role": "Referenz fuer binaeren Classifier-Code"},
    {"artifact": "data/processed/tabpfn_3_experiments/", "role": "Legacy-Cache, nicht neue Ergebnisquelle"},
    {"artifact": "data/processed/tabpfn_top10_metrics.csv", "role": "historische Top10-Metrikdatei"},
])
legacy_artifacts["exists"] = legacy_artifacts["artifact"].map(lambda p: (PROJECT_ROOT / p).exists())
legacy_artifacts.to_csv(SETUP_DIR / "legacy_artifact_overview.csv", index=False)
legacy_artifacts


,artifact,role,exists
0,src/Notebooks/12-00-01_Versuch_1_TabPFN_3.ipynb,historische lokale Top10-Tests,True
1,src/Notebooks/12-00-02_tabpfn-client_Versuch.i...,historischer API-Versuch,True
2,src/Notebooks/tabpfn_alt/12-02-02_TabPFN_Clien...,"Legacy-Learning-Curve, nur Kontext",True
3,src/Notebooks/tabpfn_alt/12-03-01_TabPFN_Class...,Referenz fuer binaeren Classifier-Code,True
4,data/processed/tabpfn_3_experiments/,"Legacy-Cache, nicht neue Ergebnisquelle",True
5,data/processed/tabpfn_top10_metrics.csv,historische Top10-Metrikdatei,True


## Ergebnis

Dieses Notebook erzeugt:
- `model_data_split_overview.csv`
- `target_distribution_by_split.csv`
- `feature_columns.csv`
- `legacy_artifact_overview.csv`
